<a href="https://colab.research.google.com/github/amilafr/algo-python-pro/blob/main/M6L4_The_'Maze'_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M6L4 The 'Maze' game

[PPT M6L4 IND](https://docs.google.com/presentation/d/1ykZUEsSggj5Qj0Kc4f5ngxFDAZIZoNwdQ5dFnjZvqDw/edit?usp=sharing)

In [ ]:
# import library
from pygame import *

# bikin window
window = display.set_mode((700, 500)) # lebar, tinggi
# ngasih judul aplikasi
display.set_caption('Gameee')

back = (119, 210, 223)

# bikin variable image
# load file gambar
bg = image.load('images/amila-fira.png')
# resize
bg = transform.scale(bg, (700, 500))

# template gamesprite
class GameSprite(sprite.Sprite):
    # constructor --> fungsi yg pasti dipanggil
    def __init__(self, picture, w, h, x, y):
        super().__init__()

        # properti
        # load gambar
        self.image = transform.scale(image.load(picture), (w, h))
        # object rectangle
        self.rect = self.image.get_rect()
        # posisi
        self.rect.x = x
        self.rect.y = y

    # nampilin sprite
    def reset(self):
        window.blit(self.image, (self.rect.x, self.rect.y))

class Player(GameSprite):
    # constructor
    def __init__(self, picture, w, h, x, y, x_speed, y_speed):
        super().__init__(picture, w, h, x, y)
        # seberapa jauh pindahnya
        self.x_speed = x_speed
        self.y_speed = y_speed

    # untuk bergerak
    def update(self):
        # koordinatnya diubah/ditambah

        # kanan-kiri
        self.rect.x += self.x_speed
        platform_touched = sprite.spritecollide(self, barriers, False) # sprite-grup
        if self.x_speed > 0: # gerak ke kanan
            for p in platform_touched:
                self.rect.right = min(self.rect.right, p.rect.left)
        elif self.x_speed < 0: # gerak ke kiri
            for p in platform_touched:
                self.rect.left = max(self.rect.left, p.rect.right)

        # atas-bawah
        self.rect.y += self.y_speed
        platform_touched = sprite.spritecollide(self, barriers, False) # sprite-grup
        if self.y_speed > 0: # gerak ke bawah
            for p in platform_touched:
                self.rect.bottom = min(self.rect.bottom, p.rect.top)
        elif self.y_speed < 0: # gerak ke atas
            for p in platform_touched:
                self.rect.top = max(self.rect.top, p.rect.bottom)

    # playernya menembak
    def fire(self):
        # bikin objek peluru/bullet
        bullet = Bullet(picture='bullet.png', x=self.rect.right, y=self.rect.centery, w=15, h=15, speed=15)
        bullets.add(bullet)

# kelas Musuh
class Enemy(GameSprite):
    # constructor
    def __init__(self, picture, w, h, x, y, speed):
        super().__init__(picture, w, h, x, y)
        # seberapa jauh pindahnya
        self.speed = speed

        self.side = 'left' # variabel sisi

    # gerak kanan-kiri
    def update(self):
        # kalo udah natap tembok --> gerak ke kanan
        if self.rect.x <= 390:
            self.side = 'right'
        # natap pinggir --> gerak ke kiri
        if self.rect.x >= 700 - 80:
            self.side = 'left'

        # untuk gerak sesuai side
        if self.side == 'left': # gerak ke kiri
            self.rect.x -= self.speed # koord x berkurang
        else: # gerak ke kanan
            self.rect.x += self.speed

# kelas peluru
class Bullet(GameSprite):
    # constructor
    def __init__(self, picture, w, h, x, y, speed):
        super().__init__(picture, w, h, x, y)
        self.speed = speed

    # gerakan bullet
    def update(self):
        self.rect.x += self.speed # bergerak ke kanan
        if self.rect.x > 700 : # udah sampe pinggir
            self.kill() # hilang

# bikin objek
# player = GameSprite(picture='ghost.png', x=350, y=250, w=80, h=80)
player = Player(picture='ghost.png', x=100, y=400, w=80, h=80, x_speed=0, y_speed=0)

# enemy dan finish
# enemy = GameSprite(picture='halloween.png', x=550, y=50, w=80, h=80)
enemy = Enemy(picture='halloween.png', x=550, y=50, w=80, h=80, speed=5)
enemies = sprite.Group()
enemies.add(enemy)

final = GameSprite(picture='pacman.png', x=550, y=350, w=80, h=80)

# tembok
w1 = GameSprite(picture='platform2.png', x=350, y=100, w=50, h=400)
w2 = GameSprite(picture='platform2.png', x=100, y=200, h=50, w=400)

# group untuk sprite tembok
barriers = sprite.Group() # bikin grup
barriers.add(w1) # nambahin sprite ke grup
barriers.add(w2)

bullets = sprite.Group()

'''loop game'''
run = True # close
finish = False # selesai/belum game-nya

while run:
    time.delay(50) # fps

    # get event
    for e in event.get():
        # mau make yg mana
        if e.type == QUIT: # menekan tombol close
            run = False
        elif e.type == KEYDOWN: # klik tombol di keyboard
            if e.key == K_UP: # jika yang ditekan tombol panah atas
                player.y_speed = -5
            elif e.key == K_DOWN:
                player.y_speed = 5
            elif e.key == K_RIGHT:
                player.x_speed = 5
            elif e.key == K_LEFT:
                player.x_speed = -5
            # nembak
            elif e.key == K_SPACE:
                player.fire()
        elif e.type == KEYUP: # udah gak mencet tombol keyboard lagi
            if e.key == K_UP: # jika yang ditekan tombol panah atas
                player.y_speed = 0
            elif e.key == K_DOWN:
                player.y_speed = 0
            elif e.key == K_RIGHT:
                player.x_speed = 0
            elif e.key == K_LEFT:
                player.x_speed = 0

    if not finish:
        # nampilin gambarnya di posisi
        # window.blit(bg, (0,0)) # x,y
        window.fill(back)

        # nampilin sprite
        player.reset()

        # enemy.reset()
        final.reset()

        # w1.reset()
        # w2.reset()
        # nampilin grup sprite
        barriers.draw(window)
        bullets.draw(window)
        enemies.draw(window)

        # bergerak
        player.update()
        # enemy.update()
        enemies.update()
        bullets.update()

        # tabrakan peluru - tembok
        sprite.groupcollide(bullets, barriers, True, False)
        # tabrakan peluru dan musuh
        sprite.groupcollide(bullets, enemies, True, True)

        # if sprite.groupcollide(bullets, enemies, True, True):
        #     enemies.kill()

        # kondisi menang
        if sprite.collide_rect(player, final): # ngecek player dan final bersentuhan atau tidak
            finish = True
            img = image.load('win.jpg') # load image
            img = transform.scale(img,(700, 500)) # resize
            window.blit(img, (0,0)) # nampilin gambar

        # kalah
        if sprite.spritecollide(player, enemies, False): # sprite-group
            finish = True
            img = image.load('lose.jpeg') # load image
            img = transform.scale(img,(700, 500)) # resize
            window.blit(img, (0,0)) # nampilin gambar


    display.update() # update frame